In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('ticks')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25
two_colors = ['#6384B3', '#F08743']


In [ ]:
data = pd.read_pickle('./full_data.pkl')
data = data.loc[data['age']>18]
print(data.shape[0])
data_add = pd.read_pickle('./ini_entropy.pkl')
data_beh = pd.read_pickle('./user_beh.pkl')

data = data.drop(columns=['expose_num_ex_search1', 'attention_span1', 'watched_search1','top10avg'])
data = data.merge(data_add, on='user_id')
data = data.merge(data_beh, on='user_id')

In [ ]:
def age_range_mapping(x):
    if 18<=x<=23:
        return '18-23'
    elif 24<=x<=30:
        return '24-30'
    elif 31<=x<=40:
        return '31-40'
    elif 41<=x<=50:
        return '41-50'
    elif 51<=x<=60:
        return '51-60'
    elif x>=61:
        return '60+'
    else:
        return None
    
def get_outliers(data, multiplier=1.5):
    # Calculate Q1 (25th percentile) and Q3 (75th percentile).
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)

    # Calculate the interquartile range.
    IQR = Q3 - Q1

    # Define the lower and upper outlier bounds.
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR

    return lower_bound, upper_bound


def bernoulli_ci(values):
    import statsmodels.api as sm
    n = len(values)
    p = values.mean()
    conf_int = sm.stats.proportion_confint(p * n, n, method='normal')
    return conf_int

In [ ]:
data['age_range'] = data['age'].transform(lambda x: age_range_mapping(x))
data['activeness'] = data.apply(lambda x: x['collect1']+x['forward1']+x['comment1'], axis=1)

data = data.dropna()
data = data.loc[data['gender']!='UNKNOWN']
data = data.loc[data['fre_community_level_eng']!='unknown']
data = data.loc[data['fre_city_level_eng']!='unknown']
data = data.loc[data['mod_price_range']!='UNKNOWN']
data = data.loc[data['mod_price_range']!='UNKNOWN']
print(data.shape)


data_orig = data.copy()

s0_min, s0_max = get_outliers(data_orig['watch_entropy_1st'])
print('s0', s0_min, s0_max)
data  = data.loc[data['watch_entropy_1st']>=s0_min]
data  = data.loc[data['watch_entropy_1st']<=s0_max]

activeness_min, activeness_max = get_outliers(data_orig['activeness'])
print('activeness', activeness_min, activeness_max)
data  = data.loc[data['activeness']>=activeness_min]
data  = data.loc[data['activeness']<=activeness_max]


search_min, search_max = get_outliers(data_orig['watched_search1'])
print('search', search_min, search_max)
data  = data.loc[data['watched_search1']>=search_min]
data  = data.loc[data['watched_search1']<=search_max]


attention_min, attention_max = get_outliers(data_orig['attention_span1'])
print('attention', attention_min, attention_max)
data  = data.loc[data['attention_span1']>=attention_min]
data  = data.loc[data['attention_span1']<=attention_max]


top10avg_min,top10avg_max = get_outliers(data_orig['top10avg'])
print('top10avg', top10avg_min,top10avg_max)
data  = data.loc[data['top10avg']>=top10avg_min]
data  = data.loc[data['top10avg']<=top10avg_max]

In [ ]:
data_orig = []
print(data.shape)
data = data[['user_id','m_ratio','gender', 'age' ,'age_range','fre_city_level_eng', 'fre_community_level_eng', 'mod_price_range',
            'activeness', 'watched_search1', 'attention_span1', 'top10avg', 'entertainment_ratio',
            'knowledge_ratio', 'news_ratio','watch_entropy_1st']]

In [ ]:
from scipy.stats import t,sem,norm
data_std = np.std(data['m_ratio'])
data_avg = np.mean(data['m_ratio'])

# Tail probability alpha.
alpha = 0.2

# Calculate the normal quantile.
z_value = norm.ppf(alpha/2)

# Print the result.
print("Entropy bounds:", 1-z_value*data_std, 1+z_value*data_std)
data['indicator'] = data['m_ratio'].transform(lambda x: 1 if x<1+z_value*data_std else 0)

In [ ]:
data['watched_search1'] =  data['watched_search1'].transform(lambda x: x/5)
data['activeness'] =  data['activeness'].transform(lambda x: x/5)


In [ ]:
q_num1 = 10
q_num2 = 10
data['search_per_hour_q'] = pd.qcut(data['watched_search1'], q_num1, labels=False)
data['activeness_q'] = pd.qcut(data['activeness'], q_num1, labels=False)
data['attention_span_q'] = pd.qcut(data['attention_span1'], q_num2, labels=False)
data['top10avg_q'] = pd.qcut(data['top10avg'], q_num2, labels=False)

#data_tmp['explore_q'] = data_tmp.apply(lambda x: int((x['search_per_hour_q']+x['explored_per_hour_q'])/2), axis=1)
data['explore_q'] = data.apply(lambda x: int((x['search_per_hour_q']+x['activeness_q'])/2), axis=1)
data['exploit_q'] = data.apply(lambda x: int((x['attention_span_q']+x['top10avg_q'])/2), axis=1)


In [ ]:
data_groupss = data.groupby(['explore_q','exploit_q']).agg({'indicator':['sum','mean','count']}).reset_index()

In [ ]:
mat = np.zeros((q_num1,q_num2))
mat_cnt = np.zeros((q_num1,q_num2))
mat_in = np.zeros((q_num1,q_num2))
mat_out = np.zeros((q_num1,q_num2))

for i in range(q_num1):
    for j in range(q_num2):
        mat[i,j] = None if data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator'].shape[0]==0 else data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator']['mean']
        mat_cnt[i,j] = 0 if data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator'].shape[0]==0 else data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator']['count']
        mat_in[i,j] = 0 if data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator'].shape[0]==0 else data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator']['sum']
        mat_out[i,j] = mat_cnt[i,j]-mat_in[i,j]

In [ ]:
plt.figure(figsize=(6,6))
sns.heatmap(mat, center=np.mean(data['indicator']), vmin=np.min(mat), vmax=np.max(mat), cmap=sns.color_palette("vlag_r", as_cmap=True), cbar=False,cbar_kws={'label':'Ratio for Deep ICs'},square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel('Degree of Exploration')
plt.xlabel('Degree of Exploitation')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)  # Set the spine width.
plt.savefig('./figures/figure2/heatmap.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(mat, center=np.mean(data['indicator']), vmin=np.min(mat), vmax=np.max(mat), cmap=sns.color_palette("vlag_r", as_cmap=True), fmt='.2f',annot=True,
            cbar_kws={'label':'Propensity for Deep ICs'},square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel('Degree of Exploration')
plt.xlabel('Degree of Exploitation')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)  # Set the spine width.


In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(mat_cnt).invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel('Degree of Exploration')
plt.xlabel('Degree of Exploitation')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)  # Set the spine width.

In [ ]:
#(#explore, #exploit)
import pickle
results = []
for i in range(10):
    for j in range(10):
        results.append((i,j,mat_cnt[i,j]))
with open('./processed_data/count_explore_exploit.pkl', 'wb') as f:
    pickle.dump(results,f)

In [ ]:
plt.figure()
sns.barplot(data, x= 'search_per_hour_q', y='indicator')
plt.ylabel('Ratio of Deep ICs')
plt.xticks(np.arange(0,10),['{}%'.format(i*10) for i in np.arange(1,11)])

In [ ]:
plt.figure()
sns.barplot(data, x= 'activeness_q', y='indicator')
plt.ylabel('Ratio of Deep ICs')
plt.xlabel('Quantile of Explored Videos per Hour')
plt.xticks(np.arange(0,10),['{}%'.format(i*10) for i in np.arange(1,11)])

In [ ]:
plt.figure()
sns.barplot(data, x= 'attention_span_q', y='indicator')
plt.ylabel('Ratio of Deep ICs')
plt.xticks(np.arange(0,10),['{}%'.format(i*10) for i in np.arange(1,11)])

In [ ]:
plt.figure()
sns.barplot(data, x= 'top10avg_q', y='indicator')
plt.ylabel('Ratio of Deep ICs')
plt.xticks(np.arange(0,10),['{}%'.format(i*10) for i in np.arange(1,11)])

In [ ]:
sns.pointplot(data, x='explore_q' , y='indicator', hue='exploit_q')

In [ ]:
sns.pointplot(data, x='exploit_q' , y='indicator', hue='explore_q')

In [ ]:
plt.figure(figsize=(8,6))
data['indicator_str'] = data['indicator'].transform(lambda x: 'Deep ICs' if x==1 else 'Others')
sns.barplot(data, x='age_range', y='watched_search1', order=['18-23','24-30','31-40','41-50','51-60','60+'], hue='indicator_str',capsize=0.2, hue_order=['Deep ICs','Others'],palette=two_colors, saturation=1)
plt.xlabel('Age')
plt.ylabel('# of search per hour')
plt.legend(title='State')
plt.savefig('./figures/figure2/search.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
plt.figure(figsize=(8,6))
data['indicator_str'] = data['indicator'].transform(lambda x: 'Deep ICs' if x==1 else 'Others')
ax = sns.barplot(data, x='age_range', y='activeness', order=['18-23','24-30','31-40','41-50','51-60','60+'], hue='indicator_str',capsize=0.2, hue_order=['Deep ICs','Others'], palette=two_colors, saturation=1)
plt.xlabel('Age')
plt.ylabel('# of actions per hour')
_lg = ax.get_legend()
_lg.remove()
plt.savefig('./figures/figure2/action.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
plt.figure(figsize=(8,6))
data['indicator_str'] = data['indicator'].transform(lambda x: 'Deep ICs' if x==1 else 'Others')
ax = sns.barplot(data, x='age_range', y='attention_span1', order=['18-23','24-30','31-40','41-50','51-60','60+'], hue='indicator_str',capsize=0.2, hue_order=['Deep ICs','Others'], palette=two_colors, saturation=1)
plt.xlabel('Age')
plt.ylabel('Avg. Watching time (consumed)')
_lg = ax.get_legend()
_lg.remove()
plt.savefig('./figures/figure2/consumed.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
plt.figure(figsize=(8,6))
data['indicator_str'] = data['indicator'].transform(lambda x: 'Deep ICs' if x==1 else 'Others')
ax = sns.barplot(data, x='age_range', y='top10avg', order=['18-23','24-30','31-40','41-50','51-60','60+'], hue='indicator_str',capsize=0.2, hue_order=['Deep ICs','Others'], palette=two_colors, saturation=1)
plt.xlabel('Age')
plt.ylabel('Avg. watching time (Top 10%)')
_lg = ax.get_legend()
_lg.remove()
plt.savefig('./figures/figure2/top10.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)


# Locate each age group in the behavioural grid.

In [ ]:
data_groupss = data.groupby(['explore_q','exploit_q','age_range']).agg({'indicator':['sum','mean','count']}).reset_index()
mat_cnt = np.zeros((q_num1,q_num2))
mat_cnt_dict = {}

min_value = 100
max_value = 0
for age in ['18-23','24-30','31-40','41-50','51-60','60+']:
    mat_cnt = np.zeros((q_num1,q_num2))
    for i in range(q_num1):
        for j in range(q_num2):
            mat_cnt[i,j] = 0 if data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)&(data_groupss['age_range']==age)]['indicator'].shape[0]==0 else data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)&(data_groupss['age_range']==age)]['indicator']['count']
    mat_cnt =  mat_cnt/np.sum(mat_cnt)
    mat_cnt_dict[age] = mat_cnt
    min_value = min(min_value, np.min(mat_cnt))
    max_value = max(max_value, np.max(mat_cnt))

age_df_tmp = data.groupby('age_range')['user_id'].count().reset_index()
mat_cnt_dict['age_dist'] = [age_df_tmp.loc[age_df_tmp['age_range']==aa]['user_id'].to_numpy()[0] for aa in ['18-23','24-30','31-40','41-50','51-60','60+']]

In [ ]:
import pickle
with open('./processed_data/age_range_all_dist.pkl', 'wb') as f:
    pickle.dump(mat_cnt_dict,f)

In [ ]:
cmap = [sns.color_palette("Blues", as_cmap=True), sns.color_palette("Greens", as_cmap=True), sns.color_palette("YlOrBr", as_cmap=True),sns.color_palette("ch:s=.75,rot=-.25", as_cmap=True)]
mat_tmp = []
for i, age in enumerate(['18-23', '31-40', '60+']):
    mat_tmp.append(mat_cnt_dict[age])
mat_tmp = np.stack(mat_tmp, axis=0)
max_indices = np.argmax(mat_tmp, axis=0)
max_value = np.max(mat_tmp)    
print(max_indices)    
plt.figure(figsize=(8,6))
for i, age in enumerate(['18-23', '31-40', '60+']):
    sns.heatmap(np.where((max_indices==i)*(mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan), vmin=1/100, vmax=max_value*0.9, cmap=cmap[i], cbar=False, alpha=1, square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel('Degree of Exploration')
plt.xlabel('Degree of Exploitation')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1.5)  # Set the spine width.
plt.savefig('./figures/figure2/heatmap_population.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
cmap = [sns.color_palette("Blues", as_cmap=True), sns.color_palette("Greens", as_cmap=True), sns.color_palette("YlOrBr", as_cmap=True),sns.color_palette("ch:s=.75,rot=-.25", as_cmap=True)]
mat_tmp = []
for i, age in enumerate(['18-23', '31-40', '60+']):
    mat_tmp.append(mat_cnt_dict[age])
mat_tmp = np.stack(mat_tmp, axis=0)
max_indices = np.argmax(mat_tmp, axis=0)
max_value = np.max(mat_tmp)    
print(max_indices)    

fig2, ax2 = plt.subplots(1,1, figsize=(8,6))
for i, age in enumerate(['18-23', '31-40', '60+']):
    fig, ax = plt.subplots(1,1, figsize=(1,6))
    sns.heatmap(np.where((max_indices==i)*(mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan), vmin=1/100, vmax=max_value*0.9, cmap=cmap[i],  alpha=1, square=True, linewidths=0.5, linecolor='gray',cbar_ax=ax, ax=ax2).invert_yaxis()
    fig.savefig('./figures/figure2/heatmap_population_cb_{}.pdf'.format(i), bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel('Degree of Exploration')
plt.xlabel('Degree of Exploitation')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1.5)  # Set the spine width.


In [ ]:
cmap = [sns.color_palette("Blues", as_cmap=True), sns.color_palette("Greens", as_cmap=True), sns.color_palette("YlOrBr", as_cmap=True),sns.color_palette("ch:s=.75,rot=-.25", as_cmap=True)]
mat_tmp = []
for i, age in enumerate(['18-23']):
    mat_tmp.append(mat_cnt_dict[age])
mat_tmp = np.stack(mat_tmp, axis=0)
max_indices = np.argmax(mat_tmp, axis=0)
max_value = np.max(mat_tmp)    
print(max_indices)    
plt.figure(figsize=(4,3))
for i, age in enumerate(['18-23']):
    sns.heatmap(np.where((mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan), vmin=1/100, vmax=max_value, cmap=cmap[i], cbar=False, alpha=1, square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# np.where((max_indices==i)*(mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan)
# plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
# plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.xticks([],[])
plt.yticks([],[])
plt.ylabel(r'Explore $\sigma$')
plt.xlabel(r'Exploit $\alpha$')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1.5)  # Set the spine width.
plt.savefig('./figures/figure3/demo_1.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
cmap = [sns.color_palette("Blues", as_cmap=True), sns.color_palette("Greens", as_cmap=True), sns.color_palette("YlOrBr", as_cmap=True),sns.color_palette("ch:s=.75,rot=-.25", as_cmap=True)]
mat_tmp = []
for i, age in enumerate(['24-30']):
    mat_tmp.append(mat_cnt_dict[age])
mat_tmp = np.stack(mat_tmp, axis=0)
max_indices = np.argmax(mat_tmp, axis=0)
max_value = np.max(mat_tmp)    
print(max_indices)    
plt.figure(figsize=(4,3))
for i, age in enumerate(['24-30']):
    sns.heatmap(np.where((mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan), vmin=1/100, vmax=max_value, cmap=cmap[i], cbar=False, alpha=1, square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# np.where((max_indices==i)*(mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan)
# plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
# plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.xticks([],[])
plt.yticks([],[])
plt.ylabel(r'Explore $\sigma$')
plt.xlabel(r'Exploit $\alpha$')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1.5)  # Set the spine width.
plt.savefig('./figures/figure3/demo_4.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
cmap = [sns.color_palette("Blues", as_cmap=True), sns.color_palette("Greens", as_cmap=True), sns.color_palette("YlOrBr", as_cmap=True),sns.color_palette("ch:s=.75,rot=-.25", as_cmap=True)]
mat_tmp = []
for i, age in enumerate(['31-40']):
    mat_tmp.append(mat_cnt_dict[age])
mat_tmp = np.stack(mat_tmp, axis=0)
max_indices = np.argmax(mat_tmp, axis=0)
max_value = np.max(mat_tmp)    
print(max_indices)    
plt.figure(figsize=(4,3))
for i, age in enumerate(['31-40']):
    i = i+1
    sns.heatmap(np.where((mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan), vmin=1/100, vmax=max_value, cmap=cmap[i], cbar=False, alpha=1, square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# np.where((max_indices==i)*(mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan)
plt.xticks([],[])
plt.yticks([],[])
plt.ylabel(r'Explore $\sigma$')
plt.xlabel(r'Exploit $\alpha$')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1.5)  # Set the spine width.
plt.savefig('./figures/figure3/demo_2.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
cmap = [sns.color_palette("Blues", as_cmap=True), sns.color_palette("Greens", as_cmap=True), sns.color_palette("YlOrBr", as_cmap=True),sns.color_palette("ch:s=.75,rot=-.25", as_cmap=True)]
mat_tmp = []
for i, age in enumerate(['41-50']):
    mat_tmp.append(mat_cnt_dict[age])
mat_tmp = np.stack(mat_tmp, axis=0)
max_indices = np.argmax(mat_tmp, axis=0)
max_value = np.max(mat_tmp)    
print(max_indices)    
plt.figure(figsize=(4,3))
for i, age in enumerate(['41-50']):
    i = i+1
    sns.heatmap(np.where((mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan), vmin=1/100, vmax=max_value, cmap=cmap[i], cbar=False, alpha=1, square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# np.where((max_indices==i)*(mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan)
plt.xticks([],[])
plt.yticks([],[])
plt.ylabel(r'Explore $\sigma$')
plt.xlabel(r'Exploit $\alpha$')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1.5)  # Set the spine width.
plt.savefig('./figures/figure3/demo_5.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
cmap = [sns.color_palette("Blues", as_cmap=True), sns.color_palette("Greens", as_cmap=True), sns.color_palette("YlOrBr", as_cmap=True),sns.color_palette("ch:s=.75,rot=-.25", as_cmap=True)]
mat_tmp = []
for i, age in enumerate(['60+']):
    mat_tmp.append(mat_cnt_dict[age])
mat_tmp = np.stack(mat_tmp, axis=0)
max_indices = np.argmax(mat_tmp, axis=0)
max_value = np.max(mat_tmp)    
print(max_indices)    
plt.figure(figsize=(4,3))
for i, age in enumerate(['60+']):
    i = i+2
    sns.heatmap(np.where((mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan), vmin=1/100, vmax=max_value, cmap=cmap[i], cbar=False, alpha=1, square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# np.where((max_indices==i)*(mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan)
plt.xticks([],[])
plt.yticks([],[])
plt.ylabel(r'Explore $\sigma$')
plt.xlabel(r'Exploit $\alpha$')
# Draw the y-axis with an arrow.
# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1.5)  # Set the spine width.
plt.savefig('./figures/figure3/demo_3.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
cmap = [sns.color_palette("Blues", as_cmap=True), sns.color_palette("Greens", as_cmap=True), sns.color_palette("YlOrBr", as_cmap=True),sns.color_palette("ch:s=.75,rot=-.25", as_cmap=True)]
mat_tmp = []
for i, age in enumerate(['51-60']):
    mat_tmp.append(mat_cnt_dict[age])
mat_tmp = np.stack(mat_tmp, axis=0)
max_indices = np.argmax(mat_tmp, axis=0)
max_value = np.max(mat_tmp)    
print(max_indices)    
plt.figure(figsize=(4,3))
for i, age in enumerate(['51-60']):
    i = i+2
    sns.heatmap(np.where((mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan), vmin=1/100, vmax=max_value, cmap=cmap[i], cbar=False, alpha=1, square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# np.where((max_indices==i)*(mat_cnt_dict[age]>1/100), mat_cnt_dict[age], np.nan)
plt.xticks([],[])
plt.yticks([],[])
plt.ylabel(r'Explore $\sigma$')
plt.xlabel(r'Exploit $\alpha$')
# Draw the y-axis with an arrow.
# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1.5)  # Set the spine width.
plt.savefig('./figures/figure3/demo_6.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
cmap = [sns.color_palette("Blues", as_cmap=True), sns.color_palette("YlOrBr", as_cmap=True), sns.color_palette("ch:s=.25,rot=-.25", as_cmap=True), sns.color_palette("ch:s=.75,rot=-.25", as_cmap=True)]
for i, age in enumerate(['18-23','24-30','31-40','41-50','51-60','60+']):
    plt.figure()
    sns.heatmap(np.where(mat_cnt_dict[age] > 1/100, mat_cnt_dict[age], np.nan), cmap=cmap[0], cbar=False, alpha=1)

# Examine recommendation entropy across age groups.

In [ ]:
data_rec_p1 = pd.read_pickle('./avg_rec_entropy_p1.pkl')
data_rec_p2 = pd.read_pickle('./avg_rec_entropy_p1.pkl')
data_rec = pd.concat([data_rec_p1, data_rec_p2])
data_rec_p1 = 0
data_rec_p2 = 0
data_rec = data_rec.groupby('user_id')['avg_num_entropy'].mean().reset_index()

data_rec = data_rec.merge(data[['user_id', 'age_range', 'indicator', 'm_ratio']], on='user_id')


In [ ]:
sns.barplot(data_rec, x='age_range', y='avg_num_entropy',order=['18-23','24-30','31-40','41-50','51-60','60+'])

In [ ]:
sns.barplot(data_rec, x='age_range', y='m_ratio',order=['18-23','24-30','31-40','41-50','51-60','60+'])